In [ ]:
"""
Project 03: Customer Segmentation & CLV
Step 01: EDA & RFM Feature Engineering
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
import os
import time
from tqdm import tqdm

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 01: EDA & RFM Feature Engineering")
print("=" * 60)
start_time = time.time()

In [ ]:
# ── 1. Load Dataset ──
csv_path = os.path.join(RAW, "online_retail_II.csv")
print(f"Loading dataset from {csv_path}...")

In [ ]:
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset not found at {csv_path}")

In [ ]:
df = pd.read_csv(csv_path, encoding="unicode_escape")
print(f"Raw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ── 2. Initial Clean ──
print("\nCleaning data...")
df.dropna(subset=["Customer ID"], inplace=True)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalPrice"] = df["Quantity"] * df["Price"]

In [ ]:
# Filter out returns and zero-price items
df = df[df["Quantity"] > 0]
df = df[df["Price"] > 0]

In [ ]:
print(f"Cleaned shape: {df.shape}")
print(f"Unique customers: {df['Customer ID'].nunique()}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

In [ ]:
# ── 3. EDA Charts ──
print("\nGenerating EDA charts...")

In [ ]:
# Revenue distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax in tqdm(axes, desc="Plotting distributions"):
    pass

In [ ]:
axes[0].hist(df["TotalPrice"], bins=100, color="#3b82f6", alpha=0.7, edgecolor="white")
axes[0].set_title("Transaction Revenue Distribution", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Total Price")
axes[0].set_ylabel("Frequency")
axes[0].set_yscale("log")

In [ ]:
customer_spend = df.groupby("Customer ID")["TotalPrice"].sum()
axes[1].hist(customer_spend, bins=100, color="#10b981", alpha=0.7, edgecolor="white")
axes[1].set_title("Customer Total Spend Distribution", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Total Spend")
axes[1].set_ylabel("Frequency")
axes[1].set_yscale("log")

In [ ]:
monthly = df.set_index("InvoiceDate").resample("M")["TotalPrice"].sum()
axes[2].plot(monthly.index, monthly.values, color="#f59e0b", linewidth=2)
axes[2].set_title("Monthly Revenue Trend", fontsize=14, fontweight="bold")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Revenue")
axes[2].tick_params(axis="x", rotation=45)

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "eda_distributions.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: eda_distributions.png")

In [ ]:
# Country analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_countries = (
    df.groupby("Country")["TotalPrice"].sum().sort_values(ascending=False).head(10)
)
axes[0].barh(top_countries.index[::-1], top_countries.values[::-1], color="#6366f1")
axes[0].set_title("Top 10 Countries by Revenue", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Total Revenue")

In [ ]:
country_counts = (
    df.groupby("Country")["Customer ID"].nunique().sort_values(ascending=False).head(10)
)
axes[1].barh(country_counts.index[::-1], country_counts.values[::-1], color="#ec4899")
axes[1].set_title("Top 10 Countries by Customer Count", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Number of Customers")

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "eda_countries.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: eda_countries.png")

In [ ]:
# ── 4. Time Split for CLV ──
split_date = pd.Timestamp("2011-06-01")
df_calib = df[df["InvoiceDate"] < split_date]
df_obs = df[df["InvoiceDate"] >= split_date]

In [ ]:
print(f"\nCalibration records: {len(df_calib)}")
print(f"Observation records: {len(df_obs)}")

In [ ]:
# ── 5. RFM Feature Engineering ──
print("\nEngineering RFM features...")
snapshot_date = split_date

In [ ]:
rfm = df_calib.groupby("Customer ID").agg(
    {
        "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
        "Invoice": "nunique",
        "TotalPrice": "sum",
    }
)

In [ ]:
rfm.rename(
    columns={
        "InvoiceDate": "Recency",
        "Invoice": "Frequency",
        "TotalPrice": "Monetary",
    },
    inplace=True,
)

In [ ]:
print(f"RFM Matrix entries: {len(rfm)}")
print(f"\nRFM Statistics:")
print(rfm.describe().round(2))

In [ ]:
# ── 6. Define CLV Target ──
target = df_obs.groupby("Customer ID")["TotalPrice"].sum()
rfm["CLV_Target"] = target.reindex(rfm.index, fill_value=0)

In [ ]:
# Save processed data
proc_path = os.path.join(PROCESSED, "rfm_clv_data.csv")
rfm.to_csv(proc_path)
print(f"\nProcessed data saved to {proc_path}")
print(f"Final shape: {rfm.shape}")
print(rfm.head())

In [ ]:
# ── 7. RFM Distribution Charts ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features = ["Recency", "Frequency", "Monetary", "CLV_Target"]
colors = ["#3b82f6", "#10b981", "#f59e0b", "#ef4444"]

In [ ]:
for i, (feat, color) in enumerate(
    tqdm(zip(features, colors), desc="Plotting RFM distributions", total=4)
):
    ax = axes[i // 2][i % 2]
    ax.hist(rfm[feat], bins=50, color=color, alpha=0.7, edgecolor="white")
    ax.set_title(f"{feat} Distribution", fontsize=13, fontweight="bold")
    ax.set_xlabel(feat)
    ax.set_ylabel("Count")
    ax.axvline(
        rfm[feat].mean(),
        color="black",
        linestyle="--",
        label=f"Mean: {rfm[feat].mean():.1f}",
    )
    ax.legend()

In [ ]:
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "rfm_distributions.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: rfm_distributions.png")

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = rfm.corr()
im = ax.imshow(corr, cmap="RdYlBu_r", aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45)
ax.set_yticklabels(corr.columns)
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=11)
plt.colorbar(im)
ax.set_title("RFM Feature Correlation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS, "rfm_correlation.png"), dpi=150, bbox_inches="tight")
plt.close()
print("Saved: rfm_correlation.png")

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 01 COMPLETE | Time: {elapsed:.1f}s | Customers: {len(rfm)}")
print(f"{'=' * 60}")